# `05_sensitivity.ipynb` — Sensitivity arms: four diagnostic robustness checks (**NO verdict authority**)

**Spec citations:** `docs/specs/2026-05-16-ai-cost-factor-model-design.md` v0.2.9 — §2.4 (sensitivity-arms table), §0.1 CORRECTIONS-K (no-verdict-authority pin), §0.8 CORRECTIONS-Z2 (Task 13 REGIME-CONDITIONAL FAIL), §0.9 CORRECTIONS-Z3 (Task 14 PARTIAL-FAIL_TO_REJECT, power-recipe pin), §0.7 CORRECTIONS-Y-9 (UTC parity), §0.1 CORRECTIONS-Y-6 (π̂ disclaimer).

**Plan:** `docs/plans/2026-05-16-ai-cost-factor-model-plan.md` Task 15.

## CORRECTIONS-K preamble (verbatim discipline)

> Each arm is **DIAGNOSTIC ONLY**. No arm has escalation authority. They cannot rescue a FAIL into a PASS or flip a PASS to a FAIL. Their role is to surface robustness or measurement-error concerns for the Task 17 verdict memo.

No arm has verdict authority. Period.

## Convergent evidence summary (entering Task 15)

| Source | Result | Direction |
|---|---|---|
| R5 PRIMARY (Task 12) | FX share of cost variance ≈ **0.003%** | small-FX |
| Z-1 multi-period (Task 16/Z) | daily/weekly/monthly all corroborate | small-FX |
| Z-2 backcast (Task 16/Z) | 2024–2025 historical TRM regime corroborates | small-FX |
| R4-S3-COP (Task 13, v0.2.8 §0.8) | **REGIME-CONDITIONAL FAIL** (reclassified — additive-identity 1.78e-15 ✓; var-ratio < 0.01 ✓) | consistency-arm only |
| R4-S3-USD (Task 14, v0.2.9 §0.9) | **PARTIAL-FAIL_TO_REJECT** (N=28 < 38 spec floor; power_contemp ≥ 0.50) | behavioral channel inelastic |

Four independent lines of evidence converge: FX-vol is empirically not a meaningful driver of cost vol for this user × this window.

## Pin disclaimers

- **π̂ (CORRECTIONS-Y-6):** `ephemeral_pi_share = 0.398977` is a calibrated constant audited from snapshot panel; not a measured time-series. Sensitivity bounded by Task 11 tornado plots.
- **Y-9 closure (v0.2.7):** Cost data is verified at **1.000000× ccusage parity** post-UTC alignment; LHS / RHS are apples-to-apples on UTC weekday cadence.
- **v0.2.9 §0.9 Z3:** Task 14 power-recipe is pinned to contemporaneous-tokens partialling (canonical); lagged-recipe is transparency-only.

## What Task 15 produces

Four arm-results (where applicable) — each a NUMBER + interpretation. The Task 17 verdict memo consults these for robustness disclosure; **the headline verdict is set by Tasks 12–14 + §0.8/§0.9 amendments, not by this notebook**.


## Cell 2 — Anti-fishing pins (declared PRE-DATA)

Constants below are pinned **before** any data read. They must not be tuned post-hoc; doing so is silent-fishing per `feedback_pathological_halt_anti_fishing_checkpoint.md`. They inherit from Tasks 13/14 §2.2.B test geometry.


In [1]:
from __future__ import annotations

import numpy as np

# ANTI-FISHING PINS — set PRE-DATA, DO NOT TUNE
LAG_PRIMARY = 1                       # §2.2.B primary lag (k=1, from Tasks 13/14)
LAG_SENSITIVITY = 5                   # §2.4 Arm-1 weekly-horizon lag
ALPHA_LEVEL = 0.05                    # two-sided test threshold (inherited from §2.2.B + CORRECTIONS-S)
TOKENS_PROXY = 'output_tok'           # consistent with Tasks 13/14
USDCOP_SERIES = 'trm_cop_per_usd'     # Banrep TRM
COST_USD = 'notional_cost_usd'        # USD-denominated (Task 14 LHS)
COST_COP = 'notional_cost_cop'        # COP-denominated (Task 13 LHS)

# §2.4 Arm-2 (model-fixed) pinned dominance threshold
MODEL_DOMINANCE_THRESHOLD = 0.80      # day kept iff one model accounts for >80% of cost

# §2.4 Arm-4 (outlier-trimmed) pinned trim level
TRIM_LOWER_QUANTILE = 0.01            # remove bottom 1% of |Δln Tokens|
TRIM_UPPER_QUANTILE = 0.99            # remove top 1% of |Δln Tokens|

# HAC bandwidth pinned PRE-DATA as floor(T^(1/3)) per CORRECTIONS-I (Andrews 1991)
def hac_bandwidth(t_lag: int) -> int:
    """Andrews (1991) rule-of-thumb HAC bandwidth: floor(T^(1/3))."""
    return int(np.floor(t_lag ** (1 / 3)))

print('=' * 64)
print('ANTI-FISHING PINS (pre-data) — Task 15 §2.4 sensitivity arms')
print('=' * 64)
print(f'LAG_PRIMARY                 = {LAG_PRIMARY}')
print(f'LAG_SENSITIVITY (Arm-1 k=5) = {LAG_SENSITIVITY}')
print(f'ALPHA_LEVEL                 = {ALPHA_LEVEL}  (two-sided, inherited §2.2.B + CORRECTIONS-S)')
print(f'TOKENS_PROXY                = {TOKENS_PROXY}')
print(f'USDCOP_SERIES               = {USDCOP_SERIES}')
print(f'COST_USD / COST_COP         = {COST_USD} / {COST_COP}')
print(f'MODEL_DOMINANCE_THRESHOLD   = {MODEL_DOMINANCE_THRESHOLD}  (Arm-2 pin)')
print(f'TRIM band                   = [{TRIM_LOWER_QUANTILE}, {TRIM_UPPER_QUANTILE}]  (Arm-4 pin)')
print(f'HAC bandwidth               = floor(T_lag^(1/3))  per CORRECTIONS-I')
print()
print('All four arms are DIAGNOSTIC ONLY per CORRECTIONS-K — no verdict authority.')


ANTI-FISHING PINS (pre-data) — Task 15 §2.4 sensitivity arms
LAG_PRIMARY                 = 1
LAG_SENSITIVITY (Arm-1 k=5) = 5
ALPHA_LEVEL                 = 0.05  (two-sided, inherited §2.2.B + CORRECTIONS-S)
TOKENS_PROXY                = output_tok
USDCOP_SERIES               = trm_cop_per_usd
COST_USD / COST_COP         = notional_cost_usd / notional_cost_cop
MODEL_DOMINANCE_THRESHOLD   = 0.8  (Arm-2 pin)
TRIM band                   = [0.01, 0.99]  (Arm-4 pin)
HAC bandwidth               = floor(T_lag^(1/3))  per CORRECTIONS-I

All four arms are DIAGNOSTIC ONLY per CORRECTIONS-K — no verdict authority.


## Shared setup — load production panel + construct shared series

Loaded once and reused across all four arms.


In [2]:
from pathlib import Path

import polars as pl

PANEL_PATH = Path('../../data/panels/notional_cost_panel.parquet')
df = pl.read_parquet(PANEL_PATH)
print(f'panel shape: {df.shape}')
print(f'date window: {df["date_utc"].min()} -> {df["date_utc"].max()}')
print(f'columns: {df.columns}')

# Full series (same recipe as Tasks 13/14)
abs_dln_cop = df[COST_COP].log().diff().abs().drop_nulls().to_numpy()
abs_dln_usd = df[COST_USD].log().diff().abs().drop_nulls().to_numpy()
abs_dln_trm = df[USDCOP_SERIES].log().diff().abs().drop_nulls().to_numpy()
abs_dln_tok = df[TOKENS_PROXY].log().diff().abs().drop_nulls().to_numpy()

T = len(abs_dln_usd)
assert len(abs_dln_trm) == T == len(abs_dln_tok) == len(abs_dln_cop), 'series-length mismatch'
print()
print(f'T (post first-diff) = {T}')
print(f'|Δln Cost^COP|  range = [{abs_dln_cop.min():.6f}, {abs_dln_cop.max():.6f}]')
print(f'|Δln Cost^USD|  range = [{abs_dln_usd.min():.6f}, {abs_dln_usd.max():.6f}]')
print(f'|Δln USDCOP|    range = [{abs_dln_trm.min():.6f}, {abs_dln_trm.max():.6f}]')
print(f'|Δln Tokens|    range = [{abs_dln_tok.min():.6f}, {abs_dln_tok.max():.6f}]')


panel shape: (29, 11)
date window: 2026-01-06 -> 2026-05-14
columns: ['date_utc', 'notional_cost_usd', 'notional_cost_cop', 'trm_cop_per_usd', 'input_tok', 'output_tok', 'cache_create_5m', 'cache_create_1h', 'cache_read', 'n_messages', 'ephemeral_pi_share']

T (post first-diff) = 28
|Δln Cost^COP|  range = [0.002468, 5.478575]
|Δln Cost^USD|  range = [0.000832, 5.483817]
|Δln USDCOP|    range = [0.000632, 0.023392]
|Δln Tokens|    range = [0.000000, 7.697189]


## Arm-1 — Lag k=5 (weekly-horizon robustness) — **DIAGNOSTIC ONLY**

### Decision-citation block (4-part)

- **Reference:** spec §2.4 Arm-1 (lag $k=5$).
- **Why:** k=5 is the pre-pinned weekly-horizon sensitivity from §2.2.B. Already computed in Tasks 13 (R4-S3-COP) + 14 (R4-S3-USD); this cell consolidates both numbers in one place for Task 17.
- **Relevance:** weekly-horizon robustness check — does the FX-vol coefficient change sign / magnitude when we move from one-trading-day to one-trading-week lag?
- **Connection:** feeds the Arm-1 row of the final summary table; **no verdict authority**.


In [3]:
import statsmodels.api as sm

print('=' * 64)
print('Arm-1 — Lag k=5 — DIAGNOSTIC ONLY (no verdict authority)')
print('=' * 64)

# Lag-aligned arrays for k=5 — same recipe as Tasks 13 + 14 Trio 3
T_lag_k5 = T - LAG_SENSITIVITY
hac_L_k5 = hac_bandwidth(T_lag_k5)
print(f'T_lag (k={LAG_SENSITIVITY}) = {T_lag_k5}')
print(f'HAC L  = floor(T_lag^(1/3)) = {hac_L_k5}')
print()

# ── R4-S3-COP k=5 ────────────────────────────────────────────────────────────
y_cop_k5 = abs_dln_cop[LAG_SENSITIVITY:]
x1_k5 = abs_dln_trm[:-LAG_SENSITIVITY]              # |Δln USDCOP|_{t-5}
x2_k5 = abs_dln_tok[:-LAG_SENSITIVITY]              # |Δln Tokens|_{t-5}
X_k5 = sm.add_constant(np.column_stack([x1_k5, x2_k5]))

model_cop_k5 = sm.OLS(y_cop_k5, X_k5).fit(cov_type='HAC', cov_kwds={'maxlags': hac_L_k5})
alpha_1_cop_k5 = float(model_cop_k5.params[1])
se_alpha_1_cop_k5 = float(model_cop_k5.bse[1])
p_cop_k5 = float(model_cop_k5.pvalues[1])
print('R4-S3-COP k=5:')
print(f'  α̂₁^COP (k=5)  = {alpha_1_cop_k5:+.6f}')
print(f'  HAC SE        = {se_alpha_1_cop_k5:.6f}')
print(f'  t-stat        = {alpha_1_cop_k5 / se_alpha_1_cop_k5:+.4f}')
print(f'  p_two_sided   = {p_cop_k5:.6f}')
print()

# ── R4-S3-USD k=5 ────────────────────────────────────────────────────────────
y_usd_k5 = abs_dln_usd[LAG_SENSITIVITY:]
model_usd_k5 = sm.OLS(y_usd_k5, X_k5).fit(cov_type='HAC', cov_kwds={'maxlags': hac_L_k5})
alpha_1_usd_k5 = float(model_usd_k5.params[1])
se_alpha_1_usd_k5 = float(model_usd_k5.bse[1])
p_usd_k5 = float(model_usd_k5.pvalues[1])
print('R4-S3-USD k=5:')
print(f'  α̂₁^USD (k=5)  = {alpha_1_usd_k5:+.6f}')
print(f'  HAC SE        = {se_alpha_1_usd_k5:.6f}')
print(f'  t-stat        = {alpha_1_usd_k5 / se_alpha_1_usd_k5:+.4f}')
print(f'  p_two_sided   = {p_usd_k5:.6f}')
print()
print('DIAGNOSTIC ONLY — these numbers DO NOT change the Task 13 REGIME-CONDITIONAL FAIL or Task 14 PARTIAL-FAIL_TO_REJECT verdicts.')


Arm-1 — Lag k=5 — DIAGNOSTIC ONLY (no verdict authority)
T_lag (k=5) = 23
HAC L  = floor(T_lag^(1/3)) = 2

R4-S3-COP k=5:
  α̂₁^COP (k=5)  = -35.887636
  HAC SE        = 30.540578
  t-stat        = -1.1751
  p_two_sided   = 0.239963

R4-S3-USD k=5:
  α̂₁^USD (k=5)  = -35.750229
  HAC SE        = 30.452271
  t-stat        = -1.1740
  p_two_sided   = 0.240405

DIAGNOSTIC ONLY — these numbers DO NOT change the Task 13 REGIME-CONDITIONAL FAIL or Task 14 PARTIAL-FAIL_TO_REJECT verdicts.


### Interpretation — Arm-1 (k=5, weekly-horizon)

Both COP and USD k=5 estimates are reported above. Convergence with the k=1 primaries from Tasks 13 + 14 reinforces the headline pattern (no FX-vol behavioral or pass-through signal at weekly horizon either); divergence would flag lag-dependent structure worth noting in the Task 17 disposition. **Per CORRECTIONS-K, this arm has no verdict authority** — both numbers are diagnostic disclosures only.


## Arm-2 — Model-fixed subset (model-mix endogeneity check) — **DIAGNOSTIC ONLY**

### Decision-citation block (4-part)

- **Reference:** spec §2.4 Arm-2 (model-fixed subset).
- **Why:** if the user mixes Claude models (Opus / Sonnet / Haiku) day-to-day, different per-token rates inject usage-pattern × pricing-table interaction into the cost series. Restricting to days where a single model dominates (>80% of daily cost) removes that source of heterogeneity.
- **Relevance:** diagnostic for whether the headline result is contaminated by model-mix endogeneity. Anti-fishing pin: `MODEL_DOMINANCE_THRESHOLD = 0.80` was set pre-data (Cell 2).
- **Connection:** feeds the Arm-2 row of the final summary table; **no verdict authority**.


In [4]:
print('=' * 64)
print('Arm-2 — Model-fixed subset — DIAGNOSTIC ONLY (no verdict authority)')
print('=' * 64)
print()
print(f'Panel columns: {df.columns}')
print()

# Check whether the production panel carries per-day per-model cost breakdown.
PER_MODEL_COLS_PRESENT = any('model' in c.lower() for c in df.columns)
print(f'Per-model breakdown columns detected? {PER_MODEL_COLS_PRESENT}')

if PER_MODEL_COLS_PRESENT:
    raise RuntimeError(
        'Per-model breakdown columns detected — Arm-2 implementation path '
        'was not anticipated; halt and update notebook.'
    )

# v0.2.9 production panel schema is DAILY-AGGREGATE only (no model dimension):
#   ['date_utc', 'notional_cost_usd', 'notional_cost_cop', 'trm_cop_per_usd',
#    'input_tok', 'output_tok', 'cache_create_5m', 'cache_create_1h',
#    'cache_read', 'n_messages', 'ephemeral_pi_share']
# Model is collapsed inside `scripts/build_notional_cost_panel.py` via
# PricingTable lookup at write-time. Recovering per-day per-model cost
# would require re-running the JSONLReader pipeline with a model-grouping
# branch — out of scope for v0.2.x (CORRECTIONS-V `ts` parameter +
# panel-schema extension deferred to v0.3+).
print()
print('Arm-2 ANALYSIS: DEFERRED to v0.3+.')
print()
print('Rationale: the v0.2.9 production panel aggregates costs at the daily')
print('level WITHOUT preserving per-model decomposition. Dominant-model share')
print('(threshold >80%) cannot be recovered from this schema without re-running')
print('the upstream JSONLReader → PricingTable pipeline with a per-model groupby')
print('branch. Per CORRECTIONS-K + anti-fishing invariants, fabricating a')
print('workaround (e.g., heuristic model imputation from token-mix) is BANNED.')
print()
print('Deferral path: v0.3+ panel-schema extension that surfaces')
print('per-day model_id → daily_cost_share dictionary as a struct column,')
print('paired with CORRECTIONS-V `ts` time-varying rate lookup.')
print()
print('Subset N would have been computed if applicable; here N_subset = N/A.')
print('No regression run; no number reported for Arm-2.')

arm2_result = 'deferred (panel schema: daily-aggregate, no model breakdown)'


Arm-2 — Model-fixed subset — DIAGNOSTIC ONLY (no verdict authority)

Panel columns: ['date_utc', 'notional_cost_usd', 'notional_cost_cop', 'trm_cop_per_usd', 'input_tok', 'output_tok', 'cache_create_5m', 'cache_create_1h', 'cache_read', 'n_messages', 'ephemeral_pi_share']

Per-model breakdown columns detected? False

Arm-2 ANALYSIS: DEFERRED to v0.3+.

Rationale: the v0.2.9 production panel aggregates costs at the daily
level WITHOUT preserving per-model decomposition. Dominant-model share
(threshold >80%) cannot be recovered from this schema without re-running
the upstream JSONLReader → PricingTable pipeline with a per-model groupby
branch. Per CORRECTIONS-K + anti-fishing invariants, fabricating a
workaround (e.g., heuristic model imputation from token-mix) is BANNED.

Deferral path: v0.3+ panel-schema extension that surfaces
per-day model_id → daily_cost_share dictionary as a struct column,
paired with CORRECTIONS-V `ts` time-varying rate lookup.

Subset N would have been computed i

### Interpretation — Arm-2 (model-fixed, deferred)

The v0.2.9 production panel aggregates costs at the daily level without preserving a per-model dimension. Per CORRECTIONS-K + anti-fishing invariants, a workaround (heuristic model imputation) is banned. Arm-2 is therefore **DEFERRED** to a v0.3+ panel-schema extension; the Task 17 verdict memo records this deferral explicitly. **Per CORRECTIONS-K this arm has no verdict authority regardless of result, applicable or not**.


## Arm-3 — Pre-/post-price-step regime split (rate-card-change robustness) — **DIAGNOSTIC ONLY**

### Decision-citation block (4-part)

- **Reference:** spec §2.4 Arm-3 (pre/post price-step regime split).
- **Why:** if Anthropic introduced a rate-card change inside the window 2026-01-06 → 2026-05-14, demand response to FX might shift across the price step. The arm splits the sample at documented LiteLLM SHA-pinned price-step dates.
- **Relevance:** rate-card-change robustness diagnostic. Anti-fishing pin: the LiteLLM SHA is locked at `e58a561caa21169fb02174148444c08509ce7028` (single snapshot; not a time series).
- **Connection:** feeds the Arm-3 row of the final summary table; **no verdict authority**.


In [5]:
import json as _json

print('=' * 64)
print('Arm-3 — Pre/post price-step regime split — DIAGNOSTIC ONLY (no verdict authority)')
print('=' * 64)
print()

LITELLM_PRICES_PATH = Path('../../data/raw/litellm_model_prices.json')
with LITELLM_PRICES_PATH.open() as f:
    litellm = _json.load(f)

print(f'LiteLLM prices file: {LITELLM_PRICES_PATH}')
print(f'LiteLLM SHA pinned:  e58a561caa21169fb02174148444c08509ce7028  (single snapshot)')
print(f'Number of models in pinned snapshot: {len(litellm)}')
print()

# Inspect whether the pinned price file carries any time-series / version metadata
sample_model = next(iter(litellm.values())) if isinstance(litellm, dict) else None
sample_keys = list(sample_model.keys())[:10] if isinstance(sample_model, dict) else []
print(f'Sample model fields (first 10): {sample_keys}')
print()

# v0.2.9 spec §3.2 + §3.5 + CORRECTIONS-O: the LiteLLM file is a single SHA
# snapshot. It contains current rates only — no historical time-series of
# price-step dates. To run Arm-3 we would need either:
#   (a) a per-day time-varying rate table (CORRECTIONS-V `ts` parameter,
#       deferred to v0.3+), OR
#   (b) an Anthropic-published rate-change calendar within 2026-01-06 →
#       2026-05-14. No such calendar is referenced in the spec corrections
#       (CORRECTIONS-V explicitly defers this to v0.3+).

print('Within-window rate-change documentation: NONE in pinned SHA.')
print()
print('Arm-3 ANALYSIS: INAPPLICABLE within current data.')
print()
print('Rationale: the LiteLLM price table is pinned to a single SHA snapshot')
print('(e58a561caa21169fb02174148444c08509ce7028, 2026-05-14). It does not')
print('expose a price-step time-series within the window 2026-01-06 → 2026-05-14.')
print('Splitting the regression at a documented price-step requires a')
print('time-varying rate lookup that is deferred to v0.3+ per CORRECTIONS-V')
print('(`PricingTable.__call__(model, ts, toks)`).')
print()
print('Per CORRECTIONS-K + anti-fishing invariants, synthesizing fictitious')
print('step dates is BANNED. No regression run; no number reported for Arm-3.')

arm3_result = 'INAPPLICABLE (single-SHA pin; no within-window rate-step time series)'


Arm-3 — Pre/post price-step regime split — DIAGNOSTIC ONLY (no verdict authority)

LiteLLM prices file: ../../data/raw/litellm_model_prices.json
LiteLLM SHA pinned:  e58a561caa21169fb02174148444c08509ce7028  (single snapshot)
Number of models in pinned snapshot: 2708

Sample model fields (first 10): ['code_interpreter_cost_per_session', 'computer_use_input_cost_per_1k_tokens', 'computer_use_output_cost_per_1k_tokens', 'deprecation_date', 'file_search_cost_per_1k_calls', 'file_search_cost_per_gb_per_day', 'input_cost_per_audio_token', 'input_cost_per_token', 'litellm_provider', 'max_input_tokens']

Within-window rate-change documentation: NONE in pinned SHA.

Arm-3 ANALYSIS: INAPPLICABLE within current data.

Rationale: the LiteLLM price table is pinned to a single SHA snapshot
(e58a561caa21169fb02174148444c08509ce7028, 2026-05-14). It does not
expose a price-step time-series within the window 2026-01-06 → 2026-05-14.
Splitting the regression at a documented price-step requires a
time-v

### Interpretation — Arm-3 (price-step split, inapplicable)

The LiteLLM price table is pinned to a single commit SHA (`e58a561caa21169fb02174148444c08509ce7028`) — it represents one snapshot, not a time-series of rate-step dates. Without documented within-window price-step dates from Anthropic, the pre/post split is undefined. Per CORRECTIONS-K + anti-fishing invariants, fabricating step dates is banned. Arm-3 is therefore **INAPPLICABLE** within the v0.2.x data architecture; deferred to v0.3+ when the `CORRECTIONS-V` time-varying rate lookup lands. **Per CORRECTIONS-K this arm has no verdict authority regardless of applicability**.


## Arm-4 — Outlier-trimmed (top/bottom 1% of $|\Delta\ln\text{Tokens}|$) — **DIAGNOSTIC ONLY**

### Decision-citation block (4-part)

- **Reference:** spec §2.4 Arm-4 (outlier-trimmed at 1%/99%).
- **Why:** single-day token spikes (e.g., a 10K-message session) can swamp the variance of $|\Delta\ln\text{Tokens}|$ and dominate the partialling regression. Trimming the top/bottom 1% of the regressor's distribution checks whether the headline result hinges on a handful of extreme days.
- **Relevance:** usage-spike sensitivity diagnostic. Anti-fishing pin: `TRIM_LOWER_QUANTILE = 0.01`, `TRIM_UPPER_QUANTILE = 0.99` — pinned pre-data (Cell 2), do not tune.
- **Connection:** feeds the Arm-4 row of the final summary table; **no verdict authority**.


In [6]:
print('=' * 64)
print('Arm-4 — Outlier-trimmed top/bottom 1% of |Δln Tokens| — DIAGNOSTIC ONLY')
print('=' * 64)
print()

# Compute quantile cut points on the FULL post-first-diff |Δln Tokens| series
# (lag-aligned trimming would shorten the lag-1 sample further; we instead
# trim the full T=28 series and then realign for k=1).
q_lo = float(np.quantile(abs_dln_tok, TRIM_LOWER_QUANTILE))
q_hi = float(np.quantile(abs_dln_tok, TRIM_UPPER_QUANTILE))
print(f'T (pre-trim, post-first-diff) = {T}')
print(f'|Δln Tokens| 1%/99% cutpoints = [{q_lo:.6f}, {q_hi:.6f}]')

# Boolean mask of in-band observations on the FULL series.
in_band_full = (abs_dln_tok >= q_lo) & (abs_dln_tok <= q_hi)
print(f'In-band (FULL series, T={T}) observations: {in_band_full.sum()} / {T}')
print()

# Lag-1 alignment: y_t requires both abs_dln_usd[t] AND abs_dln_tok[t-1],
# abs_dln_trm[t-1]. We trim the *regressor* (lagged) — drop time-t rows whose
# lagged |Δln Tokens| is in the tail. This matches §2.4 "trimmed |Δln Tokens|"
# semantics.
y_full_usd = abs_dln_usd[LAG_PRIMARY:]
x1_full = abs_dln_trm[:-LAG_PRIMARY]
x2_full = abs_dln_tok[:-LAG_PRIMARY]
T_lag = len(y_full_usd)
assert T_lag == T - LAG_PRIMARY == len(x1_full) == len(x2_full)

mask_lag = (x2_full >= q_lo) & (x2_full <= q_hi)
N_trim = int(mask_lag.sum())
print(f'T_lag (k=1, pre-trim) = {T_lag}')
print(f'N_trim (after dropping rows whose |Δln Tokens|_{{t-1}} is outside band): {N_trim}')

y_trim = y_full_usd[mask_lag]
x1_trim = x1_full[mask_lag]
x2_trim = x2_full[mask_lag]

hac_L_trim = hac_bandwidth(N_trim)
print(f'HAC L (trimmed, T={N_trim}) = floor({N_trim ** (1/3):.4f}) = {hac_L_trim}')
print()

import statsmodels.api as sm

X_trim = sm.add_constant(np.column_stack([x1_trim, x2_trim]))
model_trim = sm.OLS(y_trim, X_trim).fit(cov_type='HAC', cov_kwds={'maxlags': hac_L_trim})
alpha_1_trim = float(model_trim.params[1])
se_alpha_1_trim = float(model_trim.bse[1])
p_trim = float(model_trim.pvalues[1])
print('R4-S3-USD trimmed (k=1):')
print(f'  N_trim         = {N_trim}')
print(f'  α̂₁^USD (trim) = {alpha_1_trim:+.6f}')
print(f'  HAC SE         = {se_alpha_1_trim:.6f}')
print(f'  t-stat         = {alpha_1_trim / se_alpha_1_trim:+.4f}')
print(f'  p_two_sided    = {p_trim:.6f}')
print()
print('Headline (Task 14, full sample k=1, v0.2.9 §0.9): α̂₁^USD = -18.12, p_2s = 0.652')
print('Per CORRECTIONS-K, this arm has NO verdict authority — disclosure only.')


Arm-4 — Outlier-trimmed top/bottom 1% of |Δln Tokens| — DIAGNOSTIC ONLY

T (pre-trim, post-first-diff) = 28
|Δln Tokens| 1%/99% cutpoints = [0.007920, 7.070145]
In-band (FULL series, T=28) observations: 26 / 28

T_lag (k=1, pre-trim) = 27
N_trim (after dropping rows whose |Δln Tokens|_{t-1} is outside band): 25
HAC L (trimmed, T=25) = floor(2.9240) = 2

R4-S3-USD trimmed (k=1):
  N_trim         = 25
  α̂₁^USD (trim) = -27.122954
  HAC SE         = 42.198596
  t-stat         = -0.6427
  p_two_sided    = 0.520389

Headline (Task 14, full sample k=1, v0.2.9 §0.9): α̂₁^USD = -18.12, p_2s = 0.652
Per CORRECTIONS-K, this arm has NO verdict authority — disclosure only.


### Interpretation — Arm-4 (outlier-trimmed)

With $T = 28$ post-first-diff, the 1%/99% trim removes at most a handful of extreme days. The trimmed $\hat\alpha_1^{USD}$ point estimate and HAC p-value are reported above for comparison with the Task 14 full-sample headline ($\hat\alpha_1^{USD} = -18.12$, $p_{2s} = 0.652$). If the trimmed estimate is materially different in magnitude or sign, that is a usage-spike sensitivity flag for the Task 17 disposition memo. **Per CORRECTIONS-K this arm has no verdict authority** — both estimates are diagnostic disclosures only.


## Final summary table

Numbers are populated by the cell below from the in-memory results of arms 1–4. The Task 17 verdict memo consults this table for robustness disclosure only.


In [7]:
summary_rows = [
    {
        'arm': '1. k=5 lag',
        'spec': 'weekly-horizon (§2.4 Arm-1)',
        'result': (
            f'COP: α̂₁ = {alpha_1_cop_k5:+.4f}, p = {p_cop_k5:.4f}  |  '
            f'USD: α̂₁ = {alpha_1_usd_k5:+.4f}, p = {p_usd_k5:.4f}'
        ),
        'comparison_to_headline': (
            'vs Task 13 (R4-S3-COP k=1, REGIME-CONDITIONAL) + Task 14 '
            '(R4-S3-USD k=1: α̂₁ = -18.12, p = 0.652)'
        ),
    },
    {
        'arm': '2. Model-fixed subset',
        'spec': 'model-mix endogeneity (§2.4 Arm-2)',
        'result': arm2_result,
        'comparison_to_headline': 'n/a — deferred to v0.3+ panel-schema extension',
    },
    {
        'arm': '3. Pre/post price-step',
        'spec': 'rate-card robustness (§2.4 Arm-3)',
        'result': arm3_result,
        'comparison_to_headline': 'n/a — deferred to v0.3+ time-varying rate lookup (CORRECTIONS-V)',
    },
    {
        'arm': '4. Outlier-trimmed (1%/99%)',
        'spec': 'usage-spike sensitivity (§2.4 Arm-4)',
        'result': (
            f'USD (N_trim={N_trim}): α̂₁ = {alpha_1_trim:+.4f}, p = {p_trim:.4f}'
        ),
        'comparison_to_headline': (
            'vs Task 14 (R4-S3-USD k=1: α̂₁ = -18.12, p = 0.652)'
        ),
    },
]

print('=' * 96)
print('TASK 15 SENSITIVITY SUMMARY (§2.4 / CORRECTIONS-K — DIAGNOSTIC ONLY)')
print('=' * 96)
for row in summary_rows:
    print()
    print(f'Arm:    {row["arm"]}')
    print(f'Spec:   {row["spec"]}')
    print(f'Result: {row["result"]}')
    print(f'Vs.:    {row["comparison_to_headline"]}')
print()
print('=' * 96)
print('All four arms are DIAGNOSTIC ONLY per CORRECTIONS-K. None has verdict authority.')
print('Task 17 verdict memo consults these for robustness disclosure; the headline')
print('verdict is set by Tasks 12–14 + §0.8 (Z2) / §0.9 (Z3) amendments.')
print('=' * 96)


TASK 15 SENSITIVITY SUMMARY (§2.4 / CORRECTIONS-K — DIAGNOSTIC ONLY)

Arm:    1. k=5 lag
Spec:   weekly-horizon (§2.4 Arm-1)
Result: COP: α̂₁ = -35.8876, p = 0.2400  |  USD: α̂₁ = -35.7502, p = 0.2404
Vs.:    vs Task 13 (R4-S3-COP k=1, REGIME-CONDITIONAL) + Task 14 (R4-S3-USD k=1: α̂₁ = -18.12, p = 0.652)

Arm:    2. Model-fixed subset
Spec:   model-mix endogeneity (§2.4 Arm-2)
Result: deferred (panel schema: daily-aggregate, no model breakdown)
Vs.:    n/a — deferred to v0.3+ panel-schema extension

Arm:    3. Pre/post price-step
Spec:   rate-card robustness (§2.4 Arm-3)
Result: INAPPLICABLE (single-SHA pin; no within-window rate-step time series)
Vs.:    n/a — deferred to v0.3+ time-varying rate lookup (CORRECTIONS-V)

Arm:    4. Outlier-trimmed (1%/99%)
Spec:   usage-spike sensitivity (§2.4 Arm-4)
Result: USD (N_trim=25): α̂₁ = -27.1230, p = 0.5204
Vs.:    vs Task 14 (R4-S3-USD k=1: α̂₁ = -18.12, p = 0.652)

All four arms are DIAGNOSTIC ONLY per CORRECTIONS-K. None has verdict autho

## Final disclaimer (CORRECTIONS-K, verbatim)

> Each arm is **DIAGNOSTIC ONLY**. No arm has escalation authority. They cannot rescue a FAIL into a PASS or flip a PASS to a FAIL. Their role is to surface robustness or measurement-error concerns for the Task 17 verdict memo.

**Headline verdicts (unchanged by this notebook):**

- R5 PRIMARY (Task 12): FX share ≈ 0.003%.
- R4-S3-COP (Task 13, v0.2.8 §0.8 CORRECTIONS-Z2): **REGIME-CONDITIONAL FAIL** (additive identity passes at 1.78e-15; var-ratio < 0.01 → §2.2.A HALT-on-FAIL reclassified to "record + continue").
- R4-S3-USD (Task 14, v0.2.9 §0.9 CORRECTIONS-Z3): **PARTIAL-FAIL_TO_REJECT** (N=28 < 38 spec floor; canonical contemporaneous power ≥ 0.50).

**Arms 2 and 3 inapplicable / deferred — explicit, not workaround.** Per anti-fishing invariants, no fabricated model imputation (Arm-2) or fictitious price-step dates (Arm-3) were introduced. Both deferrals are documented above and will be revisited at v0.3+ with the CORRECTIONS-V time-varying rate lookup and a per-model panel schema.

**Anti-fishing pin reminder.** `LAG_SENSITIVITY = 5`, `MODEL_DOMINANCE_THRESHOLD = 0.80`, `TRIM = [0.01, 0.99]`, `ALPHA_LEVEL = 0.05` (two-sided), HAC bandwidth = $\lfloor T_{\text{lag}}^{1/3} \rfloor$. None tuned post-hoc.
